In [1]:
# !pip install transformers datasets scikit-learn torch -q
# !git clone https://github.com/Tisya-V/NLP-CW.git

import torch
import numpy as np
import pandas as pd
from torch import nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset
import sys
# !pip install transformers datasets scikit-learn torch -q
# !git clone https://github.com/Tisya-V/NLP-CW.git

import torch
import numpy as np
import pandas as pd
from torch import nn
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from datasets import Dataset


# import sys
sys.path.insert(0, '/content/NLP-CW/src')
# print(sys.path)
import utils


In [2]:
train_df, dev_df, test_df = utils.load_and_clean_data()

In [3]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(df):
    enc = tokenizer(
        df["text"].tolist(),
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    if "binary_label" in df.columns:
        enc["labels"] = df["binary_label"].tolist()
    return Dataset.from_dict(enc)

train_dataset = tokenize(train_df)
dev_dataset   = tokenize(dev_df)

In [4]:
labels_array = train_df["binary_label"].values
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=labels_array
)
print(f"Class weights → 0: {class_weights[0]:.3f}, 1: {class_weights[1]:.3f}")

# Move to device for use in loss
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)
device = torch.device(device)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)

In [5]:
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.get("labels")
        outputs = model(**inputs)
        logits  = outputs.get("logits")

        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss
    
from transformers import TrainerCallback

class PrinterCallback(TrainerCallback):
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            print(f"Step {state.global_step}: {logs}", flush=True)


In [6]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    # F1 of positive class only — matches the task metric
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_pcl": f1}



In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

training_args = TrainingArguments(
    output_dir          = "./model",
    num_train_epochs    = 4,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 2e-5,
    warmup_ratio        = 0.1,
    weight_decay        = 0.01,
    eval_strategy = "epoch",
    save_strategy       = "epoch",
    load_best_model_at_end = True,
    metric_for_best_model  = "f1_pcl",
    fp16 = False,
    bf16                = torch.cuda.is_available(),
    logging_steps       = 1,
    # report_to           = "none",
    # disable_tqdm        = True,
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = dev_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [PrinterCallback()],
)

trainer.train()


In [ ]:
def get_predictions(trainer, dataset):
    preds_output = trainer.predict(dataset)
    logits = preds_output.predictions
    return np.argmax(logits, axis=-1)

# Dev predictions
dev_preds = get_predictions(trainer, dev_dataset)
print(classification_report(dev_df["binary_label"].values, dev_preds, target_names=["No PCL", "PCL"]))
np.savetxt("dev.txt", dev_preds.astype(int), fmt="%d")

# Test predictions (no labels)
test_df = pd.read_csv("test.tsv", sep="\t", header=None,
                       names=["par_id", "art_id", "keyword", "country", "text"])
test_df["text"] = test_df["text"].fillna("")
test_dataset = tokenize(test_df, tokenizer)
test_preds = get_predictions(trainer, test_dataset)
np.savetxt("test.txt", test_preds.astype(int), fmt="%d")
